##Instalando Bibliotecas

In [ ]:
print("Instalando bibliotecas...")
!pip install setfit sentence-transformers datasets pandas -q

# El resto de tus imports
import pandas as pd
import torch
from datasets import Dataset
from google.colab import drive
from setfit import SetFitModel, SetFitTrainer


print("Bibliotecas instaladas y módulos importados.")

Instalando bibliotecas...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00
Bibliotecas instaladas y módulos importados.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##VERIFICAR GPU Y MONTAR GOOGLE DRIVE

In [ ]:
if not torch.cuda.is_available():
    raise SystemError("¡ERROR! No se detectó GPU. Habilita una GPU en Colab (Entorno de ejecución > Cambiar...)")
else:
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")

# Montamos Google Drive para acceder a tu CSV
drive.mount('/content/drive')
print("Google Drive montado.")

GPU detectada: NVIDIA L4
Mounted at /content/drive
Google Drive montado.


##CARGAR Y PREPARAR LOS DATOS

In [ ]:
import os
ruta_csv = "/content/drive/MyDrive/CENIDET/Tesis/DataSetR.csv"

if not os.path.exists(ruta_csv):
    raise FileNotFoundError(f"No se encontró el archivo en {ruta_csv}. Verifica la ruta.")

# Cargar el CSV con pandas
print(f"Cargando datos desde: {ruta_csv}...")
df = pd.read_csv(ruta_csv, encoding='latin1')

# Verificar que las columnas 'text' y 'label' existan
if not all(col in df.columns for col in ['text', 'label']):
    raise ValueError("El CSV debe tener las columnas 'text' y 'label'. Verifica el nombre.")

# Convertir el DataFrame de pandas a un objeto Dataset de Hugging Face
dataset = Dataset.from_pandas(df)

print(f"Dataset cargado con {len(dataset)} muestras.")
print("Ejemplo de datos:", dataset[0])

Cargando datos desde: /content/drive/MyDrive/CENIDET/Tesis/DataSetR.csv...
Dataset cargado con 535 muestras.
Ejemplo de datos: {'text': 'La paciente de 63 años, de raza blanca no hispana, con educación de preparatoria completa y dedicada al hogar (Ama de Casa). Nunca ha fumado. No refiere antecedentes familiares de cáncer de tiroides ni de otros tipos de cáncer. Su índice de masa corporal (IMC) actual se clasifica como sobrepeso (25-30), con una altura de 62 pulgadas y un peso de 150 libras. A los 20 años, su IMC era normal (18.5-25), y a los 50 años, presentaba sobrepeso. No tiene antecedentes personales de cáncer de tiroides ni de cualquier otro tipo de cáncer.', 'label': 1}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##DIVIDIR DATOS (ENTRENAMIENTO Y EVALUACIÓN)

In [ ]:
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Muestras de entrenamiento: {len(train_dataset)}")
print(f"Muestras de evaluación: {len(eval_dataset)}")

Muestras de entrenamiento: 428
Muestras de evaluación: 107


In [ ]:
print("Distribución de etiquetas en el dataset COMPLETO:")
print(df['label'].value_counts(normalize=True) * 100)

Distribución de etiquetas en el dataset COMPLETO:
label
0    50.093458
1    49.906542
Name: proportion, dtype: float64


In [ ]:
dataset = Dataset.from_pandas(df)

print("Convirtiendo 'label' a ClassLabel...")
dataset = dataset.class_encode_column("label")
# -------------------------------------------

print(f"Dataset cargado con {len(dataset)} muestras.")
# Esta línea ahora mostrará que 'label' es de tipo ClassLabel
print("Nuevas características del dataset:", dataset.features)
print("Ejemplo de datos:", dataset[0])
print("Dividiendo datos con estratificación...")
split_dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column='label'  # <--- ¡LA LÍNEA CLAVE!
)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Muestras de entrenamiento: {len(train_dataset)}")
print(f"Muestras de evaluación: {len(eval_dataset)}")

Convirtiendo 'label' a ClassLabel...


Stringifying the column:   0%|          | 0/535 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/535 [00:00<?, ? examples/s]

Dataset cargado con 535 muestras.
Nuevas características del dataset: {'text': Value('string'), 'label': ClassLabel(names=['0', '1'])}
Ejemplo de datos: {'text': 'La paciente de 63 años, de raza blanca no hispana, con educación de preparatoria completa y dedicada al hogar (Ama de Casa). Nunca ha fumado. No refiere antecedentes familiares de cáncer de tiroides ni de otros tipos de cáncer. Su índice de masa corporal (IMC) actual se clasifica como sobrepeso (25-30), con una altura de 62 pulgadas y un peso de 150 libras. A los 20 años, su IMC era normal (18.5-25), y a los 50 años, presentaba sobrepeso. No tiene antecedentes personales de cáncer de tiroides ni de cualquier otro tipo de cáncer.', 'label': 1}
Dividiendo datos con estratificación...
Muestras de entrenamiento: 428
Muestras de evaluación: 107


##CARGAR EL MODELO BASE (RigoBERTa-Clinical)

In [ ]:
modelo_id = "IIC/RigoBERTa-Clinical"
print(f"Cargando modelo base {modelo_id} desde Hugging Face...")

model = SetFitModel.from_pretrained(modelo_id)
print("Modelo base cargado.")

Cargando modelo base IIC/RigoBERTa-Clinical desde Hugging Face...


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Modelo base cargado.


##CARGAR MODELO BASE O REANUDAR CHECKPOINT

In [ ]:
from google.colab import drive
import os

# --- Configuración del Proyecto ---
modelo_id = "IIC/RigoBERTa-Clinical"
ruta_base_checkpoints = "/content/drive/MyDrive/CENIDET/Tesis/mis_checkpoints_clinicos"
num_epochs_totales = 3
# ----------------------------------

# Montar Drive (por si acaso no está)
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount('/content/drive')
    print("Google Drive montado.")

# --- Lógica para Reanudar (Resume) ---
epoch_de_inicio = 1
ruta_modelo_a_cargar = modelo_id # Por defecto, el modelo de Hugging Face

# Buscamos hacia atrás el último checkpoint válido
print("Buscando checkpoints anteriores en Google Drive...")
for i in range(num_epochs_totales, 0, -1): # (3, 2, 1)
    ruta_checkpoint_epoch = os.path.join(ruta_base_checkpoints, f"epoch_{i}")

    # Comprobamos si la carpeta existe y contiene un modelo válido
    if os.path.exists(os.path.join(ruta_checkpoint_epoch, "config.json")):
        print(f"¡Checkpoint encontrado! Reanudando desde la Época {i}.")
        ruta_modelo_a_cargar = ruta_checkpoint_epoch
        epoch_de_inicio = i + 1 # Empezaremos en la siguiente época
        break # Dejamos de buscar

if epoch_de_inicio == 1:
    print(f"No se encontraron checkpoints. Empezando desde cero con {modelo_id}.")

# Cargar el modelo (ya sea el base o el último checkpoint)
print(f"Cargando modelo desde: {ruta_modelo_a_cargar}...")
model = SetFitModel.from_pretrained(ruta_modelo_a_cargar)
print("Modelo cargado.")

Buscando checkpoints anteriores en Google Drive...
No se encontraron checkpoints. Empezando desde cero con IIC/RigoBERTa-Clinical.
Cargando modelo desde: IIC/RigoBERTa-Clinical...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Modelo cargado.


##CONFIGURAR EL ENTRENADOR (TRAINER)

In [ ]:
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
    batch_size=2,          # Tamaño de lote para el cuerpo
    num_epochs=3            # 1 época para el "cuerpo" (contrastivo) es estándar
)

/tmp/ipython-input-2719530724.py:1: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: 

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

##CONFIGURAR TRAINER PARA 1 ÉPOCA

In [ ]:
print("Configurando el SetFitTrainer...")
trainer = SetFitTrainer(
    model=model, # Este es el 'model' que acabamos de cargar
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
    batch_size=8,   #
    num_epochs=1    # <-- ¡CRUCIAL! Siempre 1, el bucle for controla el total
)

Configurando el SetFitTrainer...


/tmp/ipython-input-9252360.py:2: DeprecationWarning: `SetFitTrainer` has been deprecated and will be removed in v2.0.0 of SetFit. Please use `Trainer` instead.
  trainer = SetFitTrainer(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

##ENTRENAR EL MODELO

In [ ]:

os.environ["WANDB_DISABLED"] = "true"
print("Iniciando entrenamiento...")

trainer.train()

print("Entrenamiento completado.")

Iniciando entrenamiento...


***** Running training *****
  Num unique pairs = 17120
  Batch size = 2
  Num epochs = 3


Step,Training Loss
1,0.495200
50,0.474500


OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 902.12 MiB is free. Process 3644 has 13.86 GiB memory in use. Of the allocated memory 12.35 GiB is allocated by PyTorch, and 1.38 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

##EVALUAR EL RENDIMIENTO

In [ ]:
print("Evaluando el modelo en el set de prueba...")
metrics = trainer.evaluate()

print("\n--- ¡Resultados Finales! ---")
print(f"Precisión (Accuracy) en el set de prueba: {metrics['accuracy'] * 100:.2f}%")

***** Running evaluation *****


Evaluando el modelo en el set de prueba...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())



--- ¡Resultados Finales! ---
Precisión (Accuracy) en el set de prueba: 45.79%


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


##GUARDAR EL MODELO (EL "CHECKPOINT")

In [ ]:
ruta_guardado_final = "/content/drive/MyDrive/CENIDET/Tesis/mi_modelo_clinico_setfit_final"

# Crear el directorio si no existe
os.makedirs(ruta_guardado_final, exist_ok=True)

print(f"Guardando modelo final en Google Drive: {ruta_guardado_final}...")
model.save_pretrained(ruta_guardado_final)
# SetFit guarda todo (modelo, cabezal, config) en esa carpeta.
print("¡Modelo guardado exitosamente en Drive!")

Guardando modelo final en Google Drive: /content/drive/MyDrive/CENIDET/Tesis/mi_modelo_clinico_setfit_final...
¡Modelo guardado exitosamente en Drive!


##ENTRENAR, EVALUAR Y GUARDAR

In [ ]:
if epoch_de_inicio > num_epochs_totales:
    print("El entrenamiento ya se completó en todas las épocas.")
else:
    print(f"Iniciando bucle de entrenamiento desde la época {epoch_de_inicio}...")

    for epoch in range(epoch_de_inicio, num_epochs_totales + 1):
        print("\n" + "="*50)
        print(f"--- INICIANDO ÉPOCA {epoch}/{num_epochs_totales} ---")
        print("="*50)

        # 1. ENTRENAR (Esto tardará ~1h 20m)
        # El 'trainer' usará el objeto 'model' y lo modificará en el sitio.
        trainer.train()

        print(f"--- ÉPOCA {epoch} COMPLETADA ---")

        # 2. EVALUAR
        metrics = trainer.evaluate()
        print("\n--- Resultados de Evaluación (después de Época " + str(epoch) + ") ---")
        print(f"Precisión (Accuracy): {metrics['accuracy'] * 100:.2f}%")

        # 3. GUARDAR CHECKPOINT EN DRIVE
        # Creamos una carpeta única para esta época
        ruta_checkpoint_epoch = os.path.join(ruta_base_checkpoints, f"epoch_{epoch}")
        os.makedirs(ruta_checkpoint_epoch, exist_ok=True)

        print(f"Guardando checkpoint en: {ruta_checkpoint_epoch}...")

        # El objeto 'model' (que pasamos al trainer) ha sido actualizado
        # por trainer.train(), así que lo guardamos.
        model.save_pretrained(ruta_checkpoint_epoch)
        print("¡Checkpoint de la Época " + str(epoch) + " guardado exitosamente en Drive!")

print("\n--- ENTRENAMIENTO TOTAL COMPLETADO ---")

Iniciando bucle de entrenamiento desde la época 1...

--- INICIANDO ÉPOCA 1/3 ---


***** Running training *****
  Num unique pairs = 17120
  Batch size = 8
  Num epochs = 1
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
/usr/local/lib/python3.12/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: alextlamatini (alextlamatini-cenidet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:263: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


Step,Training Loss
1,0.248200
50,0.486400
100,0.303100
150,0.300500
200,0.294100
250,0.273200
300,0.262000
350,0.281400
400,0.264400
450,0.237300


...

Step,Training Loss
1,0.000100
50,0.003400
100,0.002500
150,0.003900
200,0.021300
250,0.067900
300,0.093600
350,0.147400
400,0.139600
450,0.110100


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/py

--- ÉPOCA 3 COMPLETADA ---

--- Resultados de Evaluación (después de Época 3) ---
Precisión (Accuracy): 76.64%
Guardando checkpoint en: /content/drive/MyDrive/CENIDET/Tesis/mis_checkpoints_clinicos/epoch_3...


/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


¡Checkpoint de la Época 3 guardado exitosamente en Drive!

--- ENTRENAMIENTO TOTAL COMPLETADO ---


##CÓMO USAR EL MODELO GUARDADO EN EL FUTURO

In [ ]:
print("\n--- Ejemplo de cómo usar tu modelo guardado ---")

# En una futura sesión de Colab, solo necesitas hacer esto:
# 1. pip install setfit
# 2. Montar Drive
from setfit import SetFitModel

# 3. Cargar el modelo desde la ruta guardada
modelo_cargado = SetFitModel.from_pretrained(ruta_checkpoint_epoch)

# 4. Hacer predicciones
reportes_nuevos = [
    "Paciente masculino de 67 años, de raza blanca no hispana, con estudios universitarios y actualmente retirado. Nunca ha fumado. No presenta antecedentes familiares de cáncer de tiroides, pero sí refiere antecedentes familiares de otros tipos de cáncer en primer grado. Su índice de masa corporal (IMC) actual se encuentra dentro del rango normal (18.5-25), con una altura de 70 pulgadas y un peso de 170 libras. Al revisar su historial, el IMC a los 20 años era normal, mientras que a los 50 años se registró en categoría de sobrepeso (25-30). No se reportan antecedentes personales de cáncer de tiroides ni de otros tipos de cáncer.", # 0
    "Paciente femenina de 57 años, de raza blanca no hispana, con educación de preparatoria completa y dedicada al hogar (Ama de Casa). La paciente nunca ha fumado. No presenta antecedentes familiares de cáncer de tiroides, pero sí antecedente familiar de otros tipos de cáncer (excluyendo piel no basal). Su índice de masa corporal (IMC) actual se clasifica como obesidad (>30), con una altura de 69 pulgadas y un peso de 270 libras. En su historial, a los 20 años presentaba sobrepeso (IMC 25-30) y a los 50 años obesidad (IMC >30). No se reportan antecedentes personales de cáncer de tiroides ni de otros tipos de cáncer."  # 1
]

predicciones = modelo_cargado.predict(reportes_nuevos)

for reporte, pred in zip(reportes_nuevos, predicciones):
    resultado = "Positivo (1)" if pred == 1 else "Negativo (0)"
    print(f"Reporte: '{reporte}' -> Predicción: {resultado}")


--- Ejemplo de cómo usar tu modelo guardado ---
Reporte: 'Paciente masculino de 67 años, de raza blanca no hispana, con estudios universitarios y actualmente retirado. Nunca ha fumado. No presenta antecedentes familiares de cáncer de tiroides, pero sí refiere antecedentes familiares de otros tipos de cáncer en primer grado. Su índice de masa corporal (IMC) actual se encuentra dentro del rango normal (18.5-25), con una altura de 70 pulgadas y un peso de 170 libras. Al revisar su historial, el IMC a los 20 años era normal, mientras que a los 50 años se registró en categoría de sobrepeso (25-30). No se reportan antecedentes personales de cáncer de tiroides ni de otros tipos de cáncer.' -> Predicción: Negativo (0)
Reporte: 'Paciente femenina de 57 años, de raza blanca no hispana, con educación de preparatoria completa y dedicada al hogar (Ama de Casa). La paciente nunca ha fumado. No presenta antecedentes familiares de cáncer de tiroides, pero sí antecedente familiar de otros tipos de cán

/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
